# DeepMIR HW2: Pop piano generation on POP909

Runs every step of the homework on Colab (GPU runtime, e.g. T4).

| Section | What it does | When to run it |
|---|---|---|
| **1. Setup** | get the code, install packages, download POP909 | **always, first** |
| 2. Pretrained checkpoints | download our trained CP (Tasks A/B) and accompaniment models | to skip training |
| 3. Training | train REMI, CP and the accompaniment model | to train from scratch (hours) |
| 4. Task A: whole-song generation | 20 songs × 32 bars from scratch | after 2 or 3 |
| 5. Task B: prompt continuation | continue the 8-bar prompts by 24 bars | after 2 or 3 |
| 6. Bonus: melody → piano accompaniment | piano part for the bonus melodies | after 2 or 3 |
| 7. Evaluation | H1 / H4 / GS / SI metrics and report figures | after 4–6 |
| 8. Save to Google Drive | copy outputs or checkpoints to Drive | optional, at the end |

Sections 4–7 each start with a **settings cell** (checkpoint, epoch, sampling): edit it, then run the cells below it.

## 1. Setup (always run first)

### 1.1 Get the code
Clones the repo and moves into `hw2/`. If you uploaded the code yourself, skip the clone and just `%cd` into the folder.

In [ ]:
!git clone https://github.com/wy11ing/DeepMIR_2026fall.git
%cd DeepMIR_2026fall/hw2

### 1.2 Install packages
Installs the pinned versions from `requirements.txt` (Colab's preinstalled torch is kept if it is new enough).

In [ ]:
!pip install -q -r requirements.txt

### 1.3 Download POP909
Unzips to `POP909/` (`training/`, `prompts/basic-b/`, `prompts/bonus/`).

In [ ]:
!gdown 1fw0qjeGzzmVkpz6aA8W3dEXygk9rX01t
!unzip -q POP909.zip

### 1.4 Weights & Biases login (only for training)
Training logs to wandb. Skip this and add `--no_wandb` to the training commands to train without it.

In [ ]:
!wandb login

### 1.5 Mount Google Drive (optional)
Needed only if your checkpoints live on Drive or you want to save outputs there (section 8).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Pretrained checkpoints (skip if you train in section 3)

Downloads the two submitted models into the folders the later sections use:

| Model | Folder | Used by |
|---|---|---|
| CP Transformer, 512 tokens, epoch 5 | `checkpoints_cp/epoch_005/` + `checkpoints_cp/tokenizer.json` | 4.2, 5.2 |
| Accompaniment, decoder-only (no chord tokens) | `checkpoints_accomp_dec/best/` | 6 |

REMI, CP-2048 and the other CP epochs are not included: train them in section 3 to run 4.1, 4.3, 5.1, 5.3 and 7.3.

In [ ]:
# CP Transformer (512 tokens, epoch 5): the zip holds taskAB_cp_epoch5/{epoch_005/, tokenizer.json}
!gdown 1OroTsLj5tb49sGeUJrzUzvfelFTBcypO -O taskAB_cp_epoch5.zip
!unzip -q -o taskAB_cp_epoch5.zip -x "__MACOSX/*" "*.DS_Store"
!mkdir -p checkpoints_cp && cp -r taskAB_cp_epoch5/. checkpoints_cp/ && rm -rf taskAB_cp_epoch5 taskAB_cp_epoch5.zip

# accompaniment model (decoder-only): the zip holds best/
!gdown 1tboLZDgIUOfruS_Vpfq_kbgwySBFW0PA -O bonus_dec.zip
!mkdir -p checkpoints_accomp_dec && unzip -q -o bonus_dec.zip -d checkpoints_accomp_dec && rm bonus_dec.zip

!ls checkpoints_cp checkpoints_cp/epoch_005 checkpoints_accomp_dec/best

## 3. Training (skip if you downloaded checkpoints)

Each model saves `best/` (lowest validation loss), `epoch_XXX/` after every epoch (REMI and CP), and `last/` (resumable state).
If Colab disconnects, run the model's **resume** cell: it continues from `last/` with the original settings.

### 3.1 REMI Transformer (Tasks A and B)

In [ ]:
!python train.py --out_dir checkpoints --run_name remi

In [ ]:
# resume
!python train.py --resume checkpoints/last

### 3.2 CP Transformer (Tasks A and B)
Two context lengths were compared: 512 compound tokens (default) and 2048.

In [ ]:
!python train_cp.py --out_dir checkpoints_cp --run_name cp-512

In [ ]:
!python train_cp.py --seq_len 2048 --out_dir checkpoints_cp_2048_rand --run_name cp-2048-random-crop

In [ ]:
# resume (use the folder of the run that stopped)
!python train_cp.py --resume checkpoints_cp/last

### 3.3 Accompaniment model (bonus)
Decoder-only model on bar-interleaved melody/piano tokens (the submitted checkpoint). Add `--chord` (and use another `--out_dir`) for the variant that predicts one chord per bar before the piano notes.

In [ ]:
!python train_accomp.py --arch dec --out_dir checkpoints_accomp_dec --run_name accomp-dec

In [ ]:
# resume
!python train_accomp.py --resume checkpoints_accomp_dec/last

## 4. Task A: whole-song generation

Generates 20 songs × 32 bars per model (temperature 1.0, top-p 0.95, no top-k). Edit `EPOCH` to choose the checkpoint (`001`, `003` or `005`).

With the downloaded checkpoints only **4.2** runs (CP, epoch 5); the others need training in section 3.

In [ ]:
EPOCH = "005"
SAMPLING = "temp1_topk0_topp095"   # only names the output folders; the sampling flags are below

### 4.1 REMI (needs training, section 3.1)

In [ ]:
!python generate.py --ckpt_dir checkpoints/epoch_{EPOCH} --tokenizer checkpoints/tokenizer.json --out_dir remi/generated_{EPOCH}_{SAMPLING} --n_samples 20 --n_bars 32 --temperature 1.0 --top_k 0 --top_p 0.95

### 4.2 CP, 512 tokens

In [ ]:
!python generate_cp.py --ckpt_dir checkpoints_cp/epoch_{EPOCH} --tokenizer checkpoints_cp/tokenizer.json --out_dir cp_transformer/generated_cp_{EPOCH}_{SAMPLING} --n_samples 20 --temperature 1.0 --top_k 0 --top_p 0.95

### 4.3 CP, 2048 tokens (random crops) (needs training, section 3.2)

In [ ]:
!python generate_cp.py --ckpt_dir checkpoints_cp_2048_rand/epoch_{EPOCH} --tokenizer checkpoints_cp_2048_rand/tokenizer.json --out_dir cp_2048/generated_cp2048_random_{EPOCH}_{SAMPLING} --n_samples 20 --temperature 1.0 --top_k 0 --top_p 0.95

## 5. Task B: prompt continuation

Continues each prompt in `POP909/prompts/basic-b`: its first 8 bars, then 24 generated bars. Each output (`001_cont.mid`, …) holds the prompt and the continuation.

With the downloaded checkpoints only **5.2** runs (CP, epoch 5).

In [ ]:
EPOCH = "005"
SAMPLING = "temp1_topk0_topp095"
PROMPTS = "POP909/prompts/basic-b"

### 5.1 REMI (needs training, section 3.1)

In [ ]:
!python continue.py --checkpoint checkpoints/epoch_{EPOCH} --tokenizer checkpoints/tokenizer.json --prompt_dir {PROMPTS} --output remi/cont_{EPOCH}_{SAMPLING} --prompt_bars 8 --n_bars 24

### 5.2 CP, 512 tokens

In [ ]:
!python generate_cp.py --ckpt_dir checkpoints_cp/epoch_{EPOCH} --tokenizer checkpoints_cp/tokenizer.json --prompt_dir {PROMPTS} --out_dir cp_transformer/cont_cp_{EPOCH}_{SAMPLING} --prompt_bars 8

### 5.3 CP, 2048 tokens (random crops) (needs training, section 3.2)

In [ ]:
!python generate_cp.py --ckpt_dir checkpoints_cp_2048_rand/epoch_{EPOCH} --tokenizer checkpoints_cp_2048_rand/tokenizer.json --prompt_dir {PROMPTS} --out_dir cp_2048/cont_cp2048_random_{EPOCH}_{SAMPLING} --prompt_bars 8

## 6. Bonus: melody → piano accompaniment

Writes a piano part for every melody in `POP909/prompts/bonus`, timed on each prompt's `beat_midi.txt`, then scores it with the TA's `eval_bonus.py`.

In [ ]:
ACCOMP_CKPT = "checkpoints_accomp_dec/best"
ACCOMP_OUT = "accomp_dec"
TEMPERATURE, TOP_P = 0.8, 0.8   # best chord accuracy in our sampling sweep (6.2)

### 6.1 Generate and evaluate

In [ ]:
!python generate_accomp.py --arch dec --ckpt_dir {ACCOMP_CKPT} --out_dir {ACCOMP_OUT} --temperature {TEMPERATURE} --top_p {TOP_P}
!python HW2_scripts/eval_bonus.py --prompt_dir POP909/prompts/bonus --gen_midi_dir {ACCOMP_OUT} --output_csv {ACCOMP_OUT}_metrics.csv

### 6.2 Sampling sweep (optional)
Generates and scores every temperature × top-p combination; the summary is `sweep.csv` in the output folder.

In [ ]:
!python generate_accomp.py --arch dec --ckpt_dir {ACCOMP_CKPT} --sweep --out_dir sweep_dec

## 7. Evaluation (Tasks A and B)

`eval_midi.py` computes H1, H4 (pitch-class entropy over 1 / 4 bars), GS (groove similarity) and SI_short / SI_mid / SI_long (structureness) for every MIDI in a folder, writes a CSV, and saves one self-similarity / scape plot per song.

### 7.1 Evaluate one folder
Set `FOLDER` to any output folder from sections 4–5.

In [ ]:
FOLDER = "cp_transformer/generated_cp_005_temp1_topk0_topp095"
!python eval_midi.py --midi_dir {FOLDER} --output_csv {FOLDER}/metrics.csv --scape_plot_dir {FOLDER}_scape_plots

### 7.2 Evaluate every output folder (optional)
Runs 7.1 on every folder sections 4–5 produced. REMI's whole-song CSV is named `generated_XXX_metrics.csv`, which is the name `report/compare_runs.py` reads.

In [ ]:
import glob, os, subprocess

for folder in sorted(glob.glob("remi/*_temp1_*") + glob.glob("cp_transformer/*_temp1_*") + glob.glob("cp_2048/*_temp1_*")):
    name = os.path.basename(folder)
    csv_name = f"{name.split('_temp')[0]}_metrics.csv" if folder.startswith("remi/generated_") else "metrics.csv"
    plots = folder.split("_temp")[0] + "_scape_plots"
    print("evaluating", folder)
    subprocess.run(["python", "eval_midi.py", "--midi_dir", folder, "--output_csv", f"{folder}/{csv_name}",
                    "--scape_plot_dir", plots], check=True)

### 7.3 Real-data reference and report figures (optional)
Crops 100 random 32-bar windows from POP909 (the same length as the generated songs), evaluates them, then builds the comparison tables and figures in `report/`. `compare_runs.py` needs epochs 001, 003 and 005 of REMI and CP-512, for both tasks (train them in section 3; the download has only CP epoch 5). `context_length.py` also needs the CP-2048 random-crop run and the earlier fixed-chunk CP-2048 outputs in `cp_2048/`; `train_cp.py` now always uses random crops, so the fixed-chunk run can't be retrained here.

In [ ]:
!python report/make_real_reference.py --out_dir report/real32
!python eval_midi.py --midi_dir report/real32 --output_csv report/real32_metrics.csv --scape_plot_dir report/real32_scape_plots
!python report/compare_runs.py
!python report/context_length.py

## 8. Save to Google Drive (optional)
Run 1.5 first. Copies any folder (outputs or checkpoints) to Drive.

In [ ]:
import os, shutil

SOURCE = "cp_transformer"                               # folder to copy
DEST = "/content/drive/MyDrive/DeepMIR/hw2"             # Drive folder to copy into

os.makedirs(DEST, exist_ok=True)
target = os.path.join(DEST, os.path.basename(SOURCE.rstrip("/")))
shutil.copytree(SOURCE, target, dirs_exist_ok=True)
print(f"copied {SOURCE} -> {target}")